# Stage 2 preflight — complete feature-family coverage

This notebook is an inventory/audit. It performs no model fitting. Its purpose is to prove which of the 15 completed feature rounds are represented as raw additive blocks, alternative encodings, controls/readouts, or historical-only representations before model selection begins.

In [1]:
from pathlib import Path
import json, pandas as pd
import plotly.express as px
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'reports/model_selection/feature_coverage.json').is_file())
report = json.loads((ROOT/'reports/model_selection/feature_coverage.json').read_text())
print('Status:', report['status'])
print('Git head:', report['git_head'])
print('Historical rounds:', report['historical_rounds'])
print('Historical variants:', report['historical_variants'])

Status: MODEL_SELECTION_COVERAGE_AUDIT_COMPLETE
Git head: c9f29976ffceff94c0746948c316a36bac80eb91
Historical rounds: 15
Historical variants: 153


## Coverage by round

A round is not considered covered merely because it appears in historical notes. We distinguish raw additive coverage from alternative encodings and readout-only coverage.

In [2]:
coverage = pd.DataFrame(report['round_dispositions'])
display(coverage)

coverage_counts = (
    coverage['coverage']
    .value_counts()
    .rename_axis('coverage')
    .reset_index(name='rounds')
)

fig = px.bar(
    coverage_counts,
    x='coverage',
    y='rounds',
    title='Feature-round coverage mode before Stage 2',
)
fig.update_layout(
    autosize=True,
    height=430,
    margin=dict(l=80, r=35, t=80, b=105),
    showlegend=False,
)
fig.update_xaxes(title='Coverage mode', automargin=True, tickangle=-20)
fig.update_yaxes(title='Number of feature rounds', automargin=True)
fig.show(renderer='plotly_mimetype')


,coverage,round,stage2,study
0,raw_additive,1,include_all,behavioral_features
1,raw_additive,2,include_all,relational_features
2,historical_not_raw_group,3,rebuild_condition_and_copy_inside_training_folds,policy_features
3,alternative_lexical_encoding,4,compare_encoding_arm,evidence_features
4,alternative_lexical_encoding,5,compare_scope_and_copy_arms,scope_lexical_features
5,density_raw_basic_via_control,6,expose_basic_and_density_in_registry,local_support_features
6,raw_additive,7,include_all,matched_support_features
7,raw_additive,8,include_all,conditioned_geometry_features
8,actor_raw_context_readout_only,9,expose_context_24_as_raw_candidate_arm,behavior_support_features
9,raw_additive,10,include_all,crossmodal_support_features


## Actual feature-bank widths

Widths are read from the completed combination inventory. Lexical width differs between policy folds because vocabulary is fitted on the eligible training population.

In [3]:
width = pd.DataFrame([
    {'family': k, 'min_columns': v['min'], 'max_columns': v['max']}
    for k, v in report['inventory'].items()
]).sort_values('max_columns', ascending=True)

display(width)

fig = px.bar(
    width,
    y='family',
    x='max_columns',
    orientation='h',
    hover_data=['min_columns'],
    title='Maximum candidate columns by compatible feature family',
)
fig.update_layout(
    autosize=True,
    height=max(560, 34 * len(width) + 150),
    margin=dict(l=205, r=40, t=85, b=65),
    showlegend=False,
)
fig.update_xaxes(title='Maximum candidate columns', automargin=True)
fig.update_yaxes(title='', automargin=True)
fig.show(renderer='plotly_mimetype')


,family,min_columns,max_columns
9,matched_pairs,12,12
8,local_density,12,12
3,conditioned_geometry,18,18
0,actor_support,24,24
13,rule_alignment,36,36
11,prototypes,36,36
2,bm25,48,48
4,consistency,48,48
10,passages,48,48
15,windows,48,48


## Stage-1 signal is diagnostic, not a hard feature cap

The readout shortlist is intentionally not treated as the final raw feature set. Low cross-fold stability is a reason to retain broader candidates for Stage 2.

In [4]:
imp = pd.DataFrame(report['stage1']['top_importance']).copy()
display(imp)

# Preserve all points but avoid permanent text-label collisions.
imp['plot_size'] = 10 + 22 * imp['selection_frequency'].astype(float)

fig = px.scatter(
    imp,
    x='mean_permutation_macro_auc_drop',
    y='mean_abs_linear_shap',
    size='plot_size',
    hover_name='readout',
    hover_data={
        'plot_size': False,
        'selection_frequency': ':.0%',
        'mean_permutation_macro_auc_drop': ':.4f',
        'mean_abs_linear_shap': ':.4f',
    },
    title='Stage-1 signals: held-out permutation, SHAP, and stability',
)
fig.update_layout(
    autosize=True,
    height=610,
    margin=dict(l=90, r=40, t=85, b=80),
)
fig.update_xaxes(title='Held-out permutation macro-AUC drop', automargin=True)
fig.update_yaxes(title='Mean absolute linear SHAP', automargin=True)
fig.show(renderer='plotly_mimetype')


,mean_abs_linear_shap,mean_permutation_macro_auc_drop,readout,selection_frequency
0,0.544175,0.105025,add_relations,0.6
1,0.407792,0.075555,add_legacy_support,0.4
2,0.386439,0.073493,without_windows,0.4
3,0.528029,0.101097,frozen_margin_all,0.2
4,0.430714,0.087355,without_bm25,0.2
5,0.349098,0.065638,add_scope,0.0
6,0.357556,0.063520,pair_bm25_passages,0.0
7,0.332105,0.058125,add_local_density,0.0
8,0.315259,0.048593,qwen_raw,0.0
9,0.264209,0.044969,add_matched_pairs,0.0


## Workspace feature-artifact census

The registered 15-round inventory is cross-checked against feature-related top-level report/run directories and current Git status. Candidate paths require reconciliation; they are not automatically treated as predictive feature families.

In [5]:
census = report['workspace_census']
print('Git branch:', census['git']['branch'])
print('Git dirty entries:', len(census['git']['dirty_entries']))
print('Unregistered report candidates:', census['unregistered_report_candidates'])
print('Unregistered run candidates:', census['unregistered_run_candidates'])
print('Feature-related notebooks:', len(census['feature_related_notebooks']))
print('Feature-related configs:', len(census['feature_related_configs']))
print('Feature-related scripts:', len(census['feature_related_scripts']))

Git branch: main
Git dirty entries: 28
Unregistered report candidates: ['competition_features', 'feature_decision', 'feature_value_audit', 'features', 'manual_feature_campaign', 'support_adaptation', 'support_context', 'support_selection']
Unregistered run candidates: ['feature_value_audit']
Feature-related notebooks: 19
Feature-related configs: 24
Feature-related scripts: 54


## Required coverage repairs before model selection

Stage 2 must explicitly restore the Round-3 policy-conditioned representation and the Round-9 context-conditional raw block. These are existing representations, not new feature engineering.

In [6]:
for item in report['coverage_gaps']['required_stage2_repairs']:
    print('-', item)
print('\nModel-selection policy:')
display(pd.DataFrame([report['model_selection_policy']]))

- Round 3 policy-conditioned lexical/behavior interactions must be rebuilt inside Stage-2 training folds and compared with norm-matched copy controls.
- Round 9 context-conditional support evidence (24 columns) must be exposed as an explicit raw candidate arm; actor-conditional 24 remains separate.

Model-selection policy:


,automatic_promotion,explainability,feature_count_selection,full_compatible_matrix_remains_candidate,hard_top_k_cap,primary_metric,secondary
0,False,raw-feature SHAP/linear contributions plus hel...,model-specific fold-internal regularization/st...,True,False,official-aligned per-rule/macro AUC developmen...,"[Brier, probability RMSE, log loss]"
